# Milestone 1: Formula 1 Podium Prediction Companion

This notebook implements the user's request in `Main.ipynb`. The attached milestone PDF is treated as assignment requirements, not as a separate instruction source. Raw CSV files are kept immutable; all cleaning and feature engineering are performed on copies in memory.

**Prediction task:** one row is one driver entering one race. The target is whether that driver finishes on the podium (`positionOrder <= 3`). Features are restricted to information available after qualifying and before race start.

## 0. Setup

The notebook is built for Kaggle, but the path resolver also supports local development. Set `F1_DATASET_DIR` if the dataset is mounted somewhere custom.

In [ ]:
import inspect
import json
import math
import os
import subprocess
import sys
import warnings
from pathlib import Path
from textwrap import dedent

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load the Relational Dataset

In [ ]:
TABLES = [
    "circuits",
    "constructors",
    "constructor_results",
    "constructor_standings",
    "drivers",
    "driver_standings",
    "lap_times",
    "pit_stops",
    "qualifying",
    "races",
    "results",
    "seasons",
    "sprint_results",
    "status",
]

def find_dataset_dir() -> Path:
    candidates = []
    env_dir = os.environ.get("F1_DATASET_DIR")
    if env_dir:
        candidates.append(Path(env_dir))

    candidates.extend(
        [
            Path("/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020"),
            Path("/kaggle/input/formula-1-world-championship-1950-2020"),
            Path("/kaggle/input/formula-1-world-championship-1950-2024"),
            Path("/kaggle/input"),
            Path.cwd(),
            Path.cwd().parent,
        ]
    )

    for candidate in candidates:
        if candidate and (candidate / "circuits.csv").exists():
            return candidate

    for root in [Path("/kaggle/input"), Path.cwd(), Path.cwd().parent]:
        if root.exists():
            for circuits_path in root.rglob("circuits.csv"):
                parent = circuits_path.parent
                required_here = sum((parent / f"{name}.csv").exists() for name in TABLES)
                if required_here >= 13:
                    return parent

    searched = "\\n".join(str(p) for p in candidates)
    raise FileNotFoundError(
        "Could not find the Formula 1 CSV directory. Set F1_DATASET_DIR or attach the Kaggle dataset. "
        f"Searched:\\n{searched}"
    )

DATA_DIR = find_dataset_dir()
print(f"Using dataset directory: {DATA_DIR}")

raw_tables = {}
for name in TABLES:
    path = DATA_DIR / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing expected table: {path}")
    raw_tables[name] = pd.read_csv(path, dtype=str, keep_default_na=False)

raw_summary = pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(df),
            "columns": df.shape[1],
            "sentinel_cells": int((df == "\\N").sum().sum()),
            "blank_cells": int((df == "").sum().sum()),
        }
        for name, df in raw_tables.items()
    ]
).sort_values("table")
display(raw_summary)

## 2. Initial EDA Before Cleaning

This first pass quantifies table sizes, sentinel missingness, temporal coverage, target balance, and the starting-grid distribution before semantic typing.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.barplot(data=raw_summary.sort_values("rows", ascending=False), y="table", x="rows", ax=axes[0, 0])
axes[0, 0].set_title("Rows per raw table")
axes[0, 0].set_xlabel("Rows")
axes[0, 0].set_ylabel("")

sentinel_plot = raw_summary.sort_values("sentinel_cells", ascending=False)
sns.barplot(data=sentinel_plot, y="table", x="sentinel_cells", ax=axes[0, 1], color="#b85042")
axes[0, 1].set_title("Literal \\\\N sentinel cells by table")
axes[0, 1].set_xlabel("Sentinel cells")
axes[0, 1].set_ylabel("")

race_years_raw = pd.to_numeric(raw_tables["races"]["year"], errors="coerce")
sns.histplot(race_years_raw, bins=min(40, max(10, race_years_raw.nunique())), ax=axes[1, 0], color="#2a6f97")
axes[1, 0].set_title("Race coverage by year")
axes[1, 0].set_xlabel("Year")

raw_position_order = pd.to_numeric(raw_tables["results"]["positionOrder"].replace("\\N", np.nan), errors="coerce")
raw_target = (raw_position_order <= 3).map({True: "Podium", False: "Not podium"})
sns.countplot(x=raw_target, order=["Podium", "Not podium"], ax=axes[1, 1], palette=["#ef476f", "#118ab2"])
axes[1, 1].set_title("Raw target balance")
axes[1, 1].set_xlabel("")
axes[1, 1].set_ylabel("Driver-race rows")

plt.tight_layout()
plt.show()

## 3. Cleaning, Semantic Typing, and Data Audits

In [ ]:
def parse_duration_to_seconds(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text in {"", "\\N"}:
        return np.nan
    text = text.replace("+", "")
    try:
        if ":" in text:
            total = 0.0
            for part in text.split(":"):
                total = total * 60 + float(part)
            return total
        return float(text)
    except ValueError:
        return np.nan

NUMERIC_COLUMNS = {
    "circuits": ["circuitId", "lat", "lng", "alt"],
    "constructors": ["constructorId"],
    "constructor_results": ["constructorResultsId", "raceId", "constructorId", "points", "status"],
    "constructor_standings": ["constructorStandingsId", "raceId", "constructorId", "points", "position", "wins"],
    "drivers": ["driverId", "number"],
    "driver_standings": ["driverStandingsId", "raceId", "driverId", "points", "position", "wins"],
    "lap_times": ["raceId", "driverId", "lap", "position", "milliseconds"],
    "pit_stops": ["raceId", "driverId", "stop", "lap", "milliseconds"],
    "qualifying": ["qualifyId", "raceId", "driverId", "constructorId", "number", "position"],
    "races": ["raceId", "year", "round", "circuitId"],
    "results": [
        "resultId",
        "raceId",
        "driverId",
        "constructorId",
        "number",
        "grid",
        "position",
        "positionOrder",
        "points",
        "laps",
        "milliseconds",
        "fastestLap",
        "rank",
        "fastestLapSpeed",
        "statusId",
    ],
    "seasons": ["year"],
    "sprint_results": [
        "resultId",
        "raceId",
        "driverId",
        "constructorId",
        "number",
        "grid",
        "position",
        "positionOrder",
        "points",
        "laps",
        "milliseconds",
        "fastestLap",
        "statusId",
    ],
    "status": ["statusId"],
}

def clean_table(name, raw_df):
    df = raw_df.replace({"\\N": np.nan, "": np.nan}).copy()

    for col in NUMERIC_COLUMNS.get(name, []):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in df.columns:
        if col == "date" or col.endswith("_date") or col == "dob":
            df[col] = pd.to_datetime(df[col], errors="coerce")

    if name == "qualifying":
        for col in ["q1", "q2", "q3"]:
            if col in df.columns:
                df[f"{col}_sec"] = df[col].map(parse_duration_to_seconds)

    if name == "lap_times" and "milliseconds" in df.columns:
        df["lap_time_sec"] = df["milliseconds"] / 1000.0

    if name == "pit_stops":
        if "milliseconds" in df.columns:
            df["pit_duration_sec"] = df["milliseconds"] / 1000.0
        elif "duration" in df.columns:
            df["pit_duration_sec"] = df["duration"].map(parse_duration_to_seconds)

    if name in {"results", "sprint_results"}:
        if "milliseconds" in df.columns:
            df["race_time_sec"] = df["milliseconds"] / 1000.0
        if "fastestLapTime" in df.columns:
            df["fastestLapTime_sec"] = df["fastestLapTime"].map(parse_duration_to_seconds)

    return df

tables = {name: clean_table(name, df) for name, df in raw_tables.items()}

cleaning_summary = pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(df),
            "columns": df.shape[1],
            "missing_cells_after_cleaning": int(df.isna().sum().sum()),
            "duplicate_rows": int(df.duplicated().sum()),
        }
        for name, df in tables.items()
    ]
).sort_values("table")
display(cleaning_summary)

In [ ]:
PRIMARY_KEYS = {
    "circuits": ["circuitId"],
    "constructors": ["constructorId"],
    "constructor_results": ["constructorResultsId"],
    "constructor_standings": ["constructorStandingsId"],
    "drivers": ["driverId"],
    "driver_standings": ["driverStandingsId"],
    "lap_times": ["raceId", "driverId", "lap"],
    "pit_stops": ["raceId", "driverId", "stop"],
    "qualifying": ["qualifyId"],
    "races": ["raceId"],
    "results": ["resultId"],
    "seasons": ["year"],
    "sprint_results": ["resultId"],
    "status": ["statusId"],
}

FOREIGN_KEYS = [
    ("races", "circuitId", "circuits", "circuitId"),
    ("results", "raceId", "races", "raceId"),
    ("results", "driverId", "drivers", "driverId"),
    ("results", "constructorId", "constructors", "constructorId"),
    ("results", "statusId", "status", "statusId"),
    ("qualifying", "raceId", "races", "raceId"),
    ("qualifying", "driverId", "drivers", "driverId"),
    ("qualifying", "constructorId", "constructors", "constructorId"),
    ("lap_times", "raceId", "races", "raceId"),
    ("lap_times", "driverId", "drivers", "driverId"),
    ("pit_stops", "raceId", "races", "raceId"),
    ("pit_stops", "driverId", "drivers", "driverId"),
    ("driver_standings", "raceId", "races", "raceId"),
    ("driver_standings", "driverId", "drivers", "driverId"),
    ("constructor_standings", "raceId", "races", "raceId"),
    ("constructor_standings", "constructorId", "constructors", "constructorId"),
    ("constructor_results", "raceId", "races", "raceId"),
    ("constructor_results", "constructorId", "constructors", "constructorId"),
    ("sprint_results", "raceId", "races", "raceId"),
    ("sprint_results", "driverId", "drivers", "driverId"),
    ("sprint_results", "constructorId", "constructors", "constructorId"),
    ("sprint_results", "statusId", "status", "statusId"),
]

def audit_primary_keys(tables_dict):
    rows = []
    for table, keys in PRIMARY_KEYS.items():
        df = tables_dict[table]
        missing_cols = [col for col in keys if col not in df.columns]
        if missing_cols:
            rows.append(
                {
                    "table": table,
                    "primary_key": ", ".join(keys),
                    "missing_key_columns": ", ".join(missing_cols),
                    "null_key_rows": np.nan,
                    "duplicate_key_rows": np.nan,
                    "passes": False,
                }
            )
            continue
        null_key_rows = int(df[keys].isna().any(axis=1).sum())
        duplicate_key_rows = int(df.duplicated(keys, keep=False).sum())
        rows.append(
            {
                "table": table,
                "primary_key": ", ".join(keys),
                "missing_key_columns": "",
                "null_key_rows": null_key_rows,
                "duplicate_key_rows": duplicate_key_rows,
                "passes": null_key_rows == 0 and duplicate_key_rows == 0,
            }
        )
    return pd.DataFrame(rows)

def audit_foreign_keys(tables_dict):
    rows = []
    for child, child_col, parent, parent_col in FOREIGN_KEYS:
        child_df = tables_dict[child]
        parent_df = tables_dict[parent]
        if child_col not in child_df.columns or parent_col not in parent_df.columns:
            rows.append(
                {
                    "child_table": child,
                    "child_column": child_col,
                    "parent_table": parent,
                    "parent_column": parent_col,
                    "checked_rows": np.nan,
                    "missing_reference_rows": np.nan,
                    "example_missing_values": "missing column",
                    "passes": False,
                }
            )
            continue

        mask_checked = child_df[child_col].notna()
        parent_values = parent_df[parent_col].dropna().unique()
        missing_mask = mask_checked & ~child_df[child_col].isin(parent_values)
        examples = child_df.loc[missing_mask, child_col].dropna().drop_duplicates().head(5).tolist()
        rows.append(
            {
                "child_table": child,
                "child_column": child_col,
                "parent_table": parent,
                "parent_column": parent_col,
                "checked_rows": int(mask_checked.sum()),
                "missing_reference_rows": int(missing_mask.sum()),
                "example_missing_values": ", ".join(map(str, examples)),
                "passes": int(missing_mask.sum()) == 0,
            }
        )
    return pd.DataFrame(rows)

primary_key_audit = audit_primary_keys(tables)
foreign_key_audit = audit_foreign_keys(tables)

display(primary_key_audit)
display(foreign_key_audit)

### Post-cleaning EDA

The next plots show how cleaning changes missingness and expose known modeling issues such as historical qualifying coverage drift.

In [ ]:
missing_long = []
for table_name, df in tables.items():
    miss = df.isna().mean().sort_values(ascending=False)
    for column, pct in miss.head(8).items():
        missing_long.append({"table": table_name, "column": column, "missing_pct": pct})
missing_long = pd.DataFrame(missing_long).sort_values("missing_pct", ascending=False).head(25)

results_with_year = tables["results"].merge(tables["races"][["raceId", "year"]], on="raceId", how="left")
results_with_year["podium"] = (results_with_year["positionOrder"] <= 3).astype(float)
results_with_year["grid_positive"] = results_with_year["grid"].where(results_with_year["grid"] > 0)

qualifying_coverage = (
    tables["results"][["raceId", "driverId"]]
    .merge(tables["races"][["raceId", "year"]], on="raceId", how="left")
    .merge(tables["qualifying"][["raceId", "driverId", "position"]], on=["raceId", "driverId"], how="left")
)
qualifying_coverage = (
    qualifying_coverage.assign(has_qualifying=lambda d: d["position"].notna())
    .groupby("year", as_index=False)["has_qualifying"]
    .mean()
)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
sns.barplot(data=missing_long, y="table", x="missing_pct", hue="column", dodge=False, ax=axes[0, 0])
axes[0, 0].set_title("Highest missingness after sentinel cleaning")
axes[0, 0].set_xlabel("Missing fraction")
axes[0, 0].legend(loc="lower right", fontsize=8)

yearly_target = results_with_year.groupby("year", as_index=False)["podium"].mean()
sns.lineplot(data=yearly_target, x="year", y="podium", ax=axes[0, 1], color="#ef476f")
axes[0, 1].set_title("Podium prevalence by season")
axes[0, 1].set_ylabel("Podium rate")

sns.lineplot(data=qualifying_coverage, x="year", y="has_qualifying", ax=axes[1, 0], color="#118ab2")
axes[1, 0].set_title("Qualifying-row coverage by season")
axes[1, 0].set_ylabel("Fraction with qualifying row")

sns.histplot(results_with_year["grid_positive"], bins=30, ax=axes[1, 1], color="#06d6a0")
axes[1, 1].set_title("Positive starting-grid positions")
axes[1, 1].set_xlabel("Grid")

plt.tight_layout()
plt.show()

## 4. Fix the Driver-Race Grain

In [ ]:
results_clean = tables["results"].copy()
duplicate_mask = results_clean.duplicated(["raceId", "driverId"], keep=False)
duplicate_driver_races = results_clean.loc[duplicate_mask].copy()

duplicate_examples = (
    duplicate_driver_races.merge(tables["races"][["raceId", "year", "name"]], on="raceId", how="left")
    .merge(tables["drivers"][["driverId", "forename", "surname"]], on="driverId", how="left")
    .sort_values(["year", "raceId", "driverId", "laps", "positionOrder"], ascending=[True, True, True, False, True])
    [["year", "name", "driverId", "forename", "surname", "resultId", "constructorId", "grid", "laps", "positionOrder", "points"]]
    .head(20)
)

display(Markdown(f"Duplicate `(raceId, driverId)` rows before handling: **{len(duplicate_driver_races)}** rows."))
display(duplicate_examples)

# Shared drives are mainly early-F1 artifacts. Keep the row where the driver completed
# the most laps; if tied, keep the better classified finish, then the lower resultId.
# This preserves the entry that best represents the driver's race exposure and target.
results_one = (
    results_clean.assign(_laps_sort=results_clean["laps"].fillna(-1))
    .sort_values(
        ["raceId", "driverId", "_laps_sort", "positionOrder", "resultId"],
        ascending=[True, True, False, True, True],
    )
    .drop_duplicates(["raceId", "driverId"], keep="first")
    .drop(columns="_laps_sort")
    .reset_index(drop=True)
)

duplicate_after = int(results_one.duplicated(["raceId", "driverId"]).sum())
display(Markdown(f"Duplicate `(raceId, driverId)` rows after handling: **{duplicate_after}**."))

## 5. Data-Engineering Questions

These analyses intentionally use joins across race, result, qualifying, driver, constructor, circuit, and status tables. The outcome is used here for retrospective analysis, but post-race fields are not used later as prediction features.

### Q1. Which circuits convert a front-row start into a podium most often, and does that ranking change after 2014?

In [ ]:
deq1 = (
    results_one[["raceId", "driverId", "grid", "positionOrder"]]
    .merge(tables["races"][["raceId", "year", "circuitId", "name"]], on="raceId", how="left")
    .rename(columns={"name": "raceName"})
    .merge(tables["circuits"][["circuitId", "name", "country"]], on="circuitId", how="left")
    .rename(columns={"name": "circuitName", "country": "circuitCountry"})
    .merge(
        tables["qualifying"][["raceId", "driverId", "position"]].rename(columns={"position": "qualifying_position"}),
        on=["raceId", "driverId"],
        how="left",
    )
)
deq1["podium"] = deq1["positionOrder"] <= 3
deq1["front_row_grid"] = deq1["grid"].between(1, 2)
deq1["front_row_qualifying"] = deq1["qualifying_position"].between(1, 2)
deq1["era"] = np.where(deq1["year"] >= 2014, "2014+ hybrid era", "pre-2014")

deq1_all_rates = (
    deq1.loc[deq1["front_row_grid"]]
    .groupby(["circuitId", "circuitName", "circuitCountry"], as_index=False)
    .agg(front_row_starts=("podium", "size"), podiums=("podium", "sum"), podium_rate=("podium", "mean"))
    .query("front_row_starts >= 6")
    .sort_values(["podium_rate", "front_row_starts"], ascending=[False, False])
)

deq1_era_rates = (
    deq1.loc[deq1["front_row_grid"]]
    .groupby(["era", "circuitId", "circuitName"], as_index=False)
    .agg(front_row_starts=("podium", "size"), podiums=("podium", "sum"), podium_rate=("podium", "mean"))
    .query("front_row_starts >= 4")
)

qual_vs_grid = (
    deq1.assign(front_row_type_grid=deq1["front_row_grid"], front_row_type_qualifying=deq1["front_row_qualifying"])
    .melt(
        id_vars=["podium"],
        value_vars=["front_row_type_grid", "front_row_type_qualifying"],
        var_name="front_row_definition",
        value_name="is_front_row",
    )
    .query("is_front_row")
    .groupby("front_row_definition", as_index=False)
    .agg(starts=("podium", "size"), podium_rate=("podium", "mean"))
)

display(deq1_all_rates.head(15))
display(qual_vs_grid)

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
sns.barplot(
    data=deq1_all_rates.head(15),
    x="podium_rate",
    y="circuitName",
    ax=axes[0],
    color="#118ab2",
)
axes[0].set_title("Top circuits: grid front row to podium")
axes[0].set_xlabel("Podium conversion rate")
axes[0].set_ylabel("")

if not deq1_era_rates.empty:
    top_circuits = deq1_era_rates.groupby("circuitName")["front_row_starts"].sum().sort_values(ascending=False).head(12).index
    sns.barplot(
        data=deq1_era_rates[deq1_era_rates["circuitName"].isin(top_circuits)],
        x="podium_rate",
        y="circuitName",
        hue="era",
        ax=axes[1],
    )
    axes[1].set_title("Front-row conversion by era")
    axes[1].set_xlabel("Podium conversion rate")
    axes[1].set_ylabel("")
else:
    axes[1].axis("off")
    axes[1].set_title("Not enough era coverage")

plt.tight_layout()
plt.show()

if deq1_all_rates.empty:
    q1_report_sentence = "No circuit met the minimum front-row sample threshold."
else:
    top_overall = deq1_all_rates.iloc[0]
    era_pivot = deq1_era_rates.pivot_table(index="circuitName", columns="era", values="podium_rate")
    if {"pre-2014", "2014+ hybrid era"}.issubset(era_pivot.columns):
        era_pivot["absolute_change"] = (era_pivot["2014+ hybrid era"] - era_pivot["pre-2014"]).abs()
        biggest_change = era_pivot.dropna().sort_values("absolute_change", ascending=False).head(1)
        change_text = (
            f"The largest pre/post-2014 change among qualifying circuits is {biggest_change.index[0]} "
            f"({biggest_change['absolute_change'].iloc[0]:.1%} absolute change)."
            if not biggest_change.empty
            else "Era comparison is limited by missing paired circuit coverage."
        )
    else:
        change_text = "Era comparison is limited by dataset coverage."
    q1_report_sentence = (
        f"The highest overall front-row grid conversion rate is {top_overall['circuitName']} "
        f"at {top_overall['podium_rate']:.1%} over {int(top_overall['front_row_starts'])} starts. "
        f"{change_text} Qualifying position and grid are tracked separately because grid can change after penalties."
    )

display(Markdown("**Answer:** " + q1_report_sentence))

### Q2. Do drivers podium more often at home after controlling for starting position?

In [ ]:
NATIONALITY_TO_COUNTRY = {
    "American": "USA",
    "Argentine": "Argentina",
    "Australian": "Australia",
    "Austrian": "Austria",
    "Belgian": "Belgium",
    "Brazilian": "Brazil",
    "British": "UK",
    "Canadian": "Canada",
    "Chilean": "Chile",
    "Chinese": "China",
    "Colombian": "Colombia",
    "Czech": "Czech Republic",
    "Danish": "Denmark",
    "Dutch": "Netherlands",
    "Finnish": "Finland",
    "French": "France",
    "German": "Germany",
    "Hungarian": "Hungary",
    "Indian": "India",
    "Indonesian": "Indonesia",
    "Irish": "Ireland",
    "Italian": "Italy",
    "Japanese": "Japan",
    "Liechtensteiner": "Liechtenstein",
    "Malaysian": "Malaysia",
    "Mexican": "Mexico",
    "Monegasque": "Monaco",
    "New Zealander": "New Zealand",
    "Polish": "Poland",
    "Portuguese": "Portugal",
    "Rhodesian": "Zimbabwe",
    "Russian": "Russia",
    "South African": "South Africa",
    "Spanish": "Spain",
    "Swedish": "Sweden",
    "Swiss": "Switzerland",
    "Thai": "Thailand",
    "Uruguayan": "Uruguay",
    "Venezuelan": "Venezuela",
}
COUNTRY_ALIASES = {
    "United States": "USA",
    "United States of America": "USA",
    "Great Britain": "UK",
    "United Kingdom": "UK",
    "UAE": "UAE",
    "United Arab Emirates": "UAE",
    "Korea": "Korea",
    "South Korea": "Korea",
}

def normalize_country(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    return COUNTRY_ALIASES.get(text, text)

def nationality_to_country(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    return COUNTRY_ALIASES.get(NATIONALITY_TO_COUNTRY.get(text, text), NATIONALITY_TO_COUNTRY.get(text, text))

deq2 = (
    results_one[["raceId", "driverId", "grid", "positionOrder"]]
    .merge(tables["races"][["raceId", "year", "circuitId"]], on="raceId", how="left")
    .merge(tables["circuits"][["circuitId", "country"]].rename(columns={"country": "circuitCountry"}), on="circuitId", how="left")
    .merge(tables["drivers"][["driverId", "nationality", "forename", "surname"]], on="driverId", how="left")
)
deq2["driverCountry"] = deq2["nationality"].map(nationality_to_country)
deq2["circuitCountryNorm"] = deq2["circuitCountry"].map(normalize_country)
deq2["home_race"] = deq2["driverCountry"] == deq2["circuitCountryNorm"]
deq2["podium"] = deq2["positionOrder"] <= 3
deq2["grid_position"] = deq2["grid"].where(deq2["grid"] > 0)
deq2 = deq2.dropna(subset=["grid_position", "home_race"])
deq2["grid_band"] = pd.cut(
    deq2["grid_position"],
    bins=[0, 3, 6, 10, 15, 20, 99],
    labels=["1-3", "4-6", "7-10", "11-15", "16-20", "21+"],
    include_lowest=True,
)

deq2_grid_summary = (
    deq2.groupby(["grid_band", "home_race"], observed=False)
    .agg(starts=("podium", "size"), podium_rate=("podium", "mean"))
    .reset_index()
)

# Controlled comparison using a simple logistic model with grid-band fixed effects.
deq2_model_data = pd.get_dummies(deq2[["home_race", "grid_band"]].dropna(), columns=["grid_band"], drop_first=True)
deq2_y = deq2.loc[deq2_model_data.index, "podium"].astype(int)
home_odds_ratio = np.nan
if deq2_y.nunique() == 2 and deq2_model_data["home_race"].nunique() == 2:
    home_model = LogisticRegression(max_iter=1000, class_weight="balanced")
    home_model.fit(deq2_model_data, deq2_y)
    home_coef = home_model.coef_[0][list(deq2_model_data.columns).index("home_race")]
    home_odds_ratio = float(np.exp(home_coef))

display(deq2_grid_summary)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=deq2_grid_summary, x="grid_band", y="podium_rate", hue="home_race", ax=ax)
ax.set_title("Home vs non-home podium rate within starting-grid bands")
ax.set_xlabel("Starting-grid band")
ax.set_ylabel("Podium rate")
ax.legend(title="Home race")
plt.tight_layout()
plt.show()

home_counts = deq2["home_race"].value_counts().to_dict()
if math.isnan(home_odds_ratio):
    q2_report_sentence = "The controlled home-race comparison could not be estimated because one class or group was absent."
else:
    direction = "higher" if home_odds_ratio > 1 else "lower"
    q2_report_sentence = (
        f"After comparing drivers within similar grid bands, the home-race indicator has an odds ratio of "
        f"{home_odds_ratio:.2f}, implying {direction} model-estimated podium odds for home starts. "
        f"Home starts are rare relative to non-home starts ({home_counts}). This is associational, not causal."
    )

display(Markdown("**Answer:** " + q2_report_sentence))

### Q3. Which constructors have the highest mechanical-retirement rate in 2014-2021 vs 2022-2024?

In [ ]:
mechanical_keywords = [
    "engine",
    "gearbox",
    "transmission",
    "clutch",
    "hydraulic",
    "electrical",
    "fuel",
    "oil",
    "brake",
    "suspension",
    "power",
    "alternator",
    "battery",
    "turbo",
    "ers",
    "mgu",
    "water",
    "radiator",
    "overheating",
    "pneumatic",
    "driveshaft",
    "wheel",
    "throttle",
    "steering",
    "exhaust",
    "ignition",
    "spark",
    "differential",
    "cv joint",
    "pump",
    "pressure",
    "distributor",
]
non_start_keywords = ["did not start", "did not qualify", "not prequalified", "withdrew", "withdrawn"]
crash_or_regulation_keywords = ["accident", "collision", "spun", "disqualified", "excluded", "damage", "driver"]

def contains_any(text, keywords):
    if pd.isna(text):
        return False
    lower = str(text).lower()
    return any(keyword in lower for keyword in keywords)

deq3 = (
    results_one[["raceId", "driverId", "constructorId", "statusId"]]
    .merge(tables["races"][["raceId", "year"]], on="raceId", how="left")
    .merge(tables["constructors"][["constructorId", "name"]].rename(columns={"name": "constructorName"}), on="constructorId", how="left")
    .merge(tables["status"], on="statusId", how="left")
)
deq3["race_start"] = ~deq3["status"].map(lambda s: contains_any(s, non_start_keywords))
deq3["mechanical_retirement"] = (
    deq3["race_start"]
    & deq3["status"].map(lambda s: contains_any(s, mechanical_keywords))
    & ~deq3["status"].map(lambda s: contains_any(s, crash_or_regulation_keywords))
)
deq3["period"] = np.nan
deq3.loc[deq3["year"].between(2014, 2021), "period"] = "2014-2021"
deq3.loc[deq3["year"].between(2022, 2024), "period"] = "2022-2024"
deq3_period = deq3.dropna(subset=["period"]).query("race_start").copy()
deq3_rates = (
    deq3_period.groupby(["period", "constructorId", "constructorName"], as_index=False)
    .agg(starts=("race_start", "size"), mechanical_retirements=("mechanical_retirement", "sum"))
    .assign(mechanical_retirement_rate=lambda d: d["mechanical_retirements"] / d["starts"])
    .query("starts >= 10")
    .sort_values(["period", "mechanical_retirement_rate", "starts"], ascending=[True, False, False])
)

statuses_used = (
    deq3.loc[deq3["mechanical_retirement"], "status"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)
display(Markdown("Mechanical status labels included: `" + "`, `".join(statuses_used[:40]) + "`"))
display(deq3_rates.groupby("period").head(10))

if deq3_rates.empty:
    q3_report_sentence = "No constructor met the minimum-start threshold in the requested periods."
else:
    top_by_period = deq3_rates.groupby("period", group_keys=False).head(1)
    q3_bits = [
        f"{row.period}: {row.constructorName} ({row.mechanical_retirement_rate:.1%}, {int(row.mechanical_retirements)}/{int(row.starts)} starts)"
        for row in top_by_period.itertuples(index=False)
    ]
    q3_report_sentence = "Highest mechanical-retirement rates by period are " + "; ".join(q3_bits) + "."

fig, ax = plt.subplots(figsize=(14, 7))
plot_q3 = deq3_rates.groupby("period", group_keys=False).head(10)
if not plot_q3.empty:
    sns.barplot(data=plot_q3, x="mechanical_retirement_rate", y="constructorName", hue="period", ax=ax)
    ax.set_title("Highest constructor mechanical-retirement rates")
    ax.set_xlabel("Mechanical-retirement rate")
    ax.set_ylabel("")
else:
    ax.axis("off")
    ax.set_title("No qualifying constructors for Q3")
plt.tight_layout()
plt.show()

display(Markdown("**Answer:** " + q3_report_sentence))

## 6. Leakage-Safe Modeling Table and Feature Engineering

In [ ]:
races_model = tables["races"][["raceId", "year", "round", "circuitId", "name", "date"]].rename(columns={"name": "raceName"})
circuits_model = tables["circuits"][["circuitId", "name", "country", "lat", "lng", "alt"]].rename(
    columns={"name": "circuitName", "country": "circuitCountry"}
)
drivers_model = tables["drivers"][["driverId", "dob", "nationality"]].rename(columns={"nationality": "driverNationality"})
constructors_model = tables["constructors"][["constructorId", "name", "nationality"]].rename(
    columns={"name": "constructorName", "nationality": "constructorNationality"}
)

model_df = (
    results_one[["raceId", "driverId", "constructorId", "grid", "positionOrder", "points"]]
    .merge(races_model, on="raceId", how="left")
    .merge(circuits_model, on="circuitId", how="left")
    .merge(drivers_model, on="driverId", how="left")
    .merge(constructors_model, on="constructorId", how="left")
)
model_df["podium"] = (model_df["positionOrder"] <= 3).astype(int)
model_df["grid_position"] = model_df["grid"].where(model_df["grid"] > 0)
model_df["grid_zero_or_unknown"] = (model_df["grid"].fillna(0) <= 0).astype(int)
model_df["driverAgeYears"] = (model_df["date"] - model_df["dob"]).dt.days / 365.25
model_df["driverCountry"] = model_df["driverNationality"].map(nationality_to_country)
model_df["circuitCountryNorm"] = model_df["circuitCountry"].map(normalize_country)
model_df["driver_home_race"] = (model_df["driverCountry"] == model_df["circuitCountryNorm"]).astype(int)

qualifying_model = tables["qualifying"][["raceId", "driverId", "position", "q1_sec", "q2_sec", "q3_sec"]].rename(
    columns={"position": "qualifying_position"}
)
qualifying_model["best_qualifying_sec"] = qualifying_model[["q1_sec", "q2_sec", "q3_sec"]].min(axis=1, skipna=True)
qualifying_model["completed_q3"] = qualifying_model["q3_sec"].notna().astype(int)

model_df = model_df.merge(qualifying_model, on=["raceId", "driverId"], how="left")
model_df["qualifying_missing"] = model_df["qualifying_position"].isna().astype(int)
model_df["qualifying_gap_to_pole"] = model_df["best_qualifying_sec"] - model_df.groupby("raceId")["best_qualifying_sec"].transform("min")
model_df["qualifying_position_delta_to_grid"] = model_df["grid_position"] - model_df["qualifying_position"]

def previous_standings(standings, entity_col, prefix):
    standing_cols = [col for col in ["points", "position", "wins"] if col in standings.columns]
    st = standings[["raceId", entity_col] + standing_cols].merge(
        tables["races"][["raceId", "year", "round", "date"]], on="raceId", how="left"
    )
    st = st.sort_values(["year", "round", "date", "raceId"])
    created = []
    for col in standing_cols:
        new_col = f"{prefix}_prev_{col}"
        st[new_col] = st.groupby(["year", entity_col])[col].shift(1)
        created.append(new_col)
    return st[["raceId", entity_col] + created]

driver_prev = previous_standings(tables["driver_standings"], "driverId", "driver")
constructor_prev = previous_standings(tables["constructor_standings"], "constructorId", "constructor")

model_df = model_df.merge(driver_prev, on=["raceId", "driverId"], how="left")
model_df = model_df.merge(constructor_prev, on=["raceId", "constructorId"], how="left")

hist = (
    results_one[["raceId", "driverId", "constructorId", "positionOrder", "points"]]
    .merge(tables["races"][["raceId", "year", "round", "date"]], on="raceId", how="left")
    .sort_values(["date", "raceId", "driverId"])
)
hist["hist_podium"] = (hist["positionOrder"] <= 3).astype(int)

def shifted_rolling_mean(series, window=5):
    return series.shift(1).rolling(window=window, min_periods=1).mean()

hist["driver_form_podium_rate_5"] = hist.groupby("driverId")["hist_podium"].transform(shifted_rolling_mean)
hist["driver_form_avg_finish_5"] = hist.groupby("driverId")["positionOrder"].transform(shifted_rolling_mean)
hist["driver_form_points_5"] = hist.groupby("driverId")["points"].transform(shifted_rolling_mean)

driver_form = hist[
    [
        "raceId",
        "driverId",
        "driver_form_podium_rate_5",
        "driver_form_avg_finish_5",
        "driver_form_points_5",
    ]
]

constructor_race_hist = (
    hist.groupby(["constructorId", "raceId", "year", "round", "date"], as_index=False)
    .agg(
        constructor_race_podium_rate=("hist_podium", "mean"),
        constructor_race_avg_finish=("positionOrder", "mean"),
        constructor_race_points=("points", "sum"),
    )
    .sort_values(["date", "raceId", "constructorId"])
)
constructor_race_hist["constructor_form_podium_rate_5"] = constructor_race_hist.groupby("constructorId")[
    "constructor_race_podium_rate"
].transform(shifted_rolling_mean)
constructor_race_hist["constructor_form_avg_finish_5"] = constructor_race_hist.groupby("constructorId")[
    "constructor_race_avg_finish"
].transform(shifted_rolling_mean)
constructor_race_hist["constructor_form_points_5"] = constructor_race_hist.groupby("constructorId")[
    "constructor_race_points"
].transform(shifted_rolling_mean)

constructor_form = constructor_race_hist[
    [
        "raceId",
        "constructorId",
        "constructor_form_podium_rate_5",
        "constructor_form_avg_finish_5",
        "constructor_form_points_5",
    ]
]

model_df = model_df.merge(driver_form, on=["raceId", "driverId"], how="left")
model_df = model_df.merge(constructor_form, on=["raceId", "constructorId"], how="left")

numeric_features = [
    "year",
    "round",
    "grid_position",
    "grid_zero_or_unknown",
    "qualifying_position",
    "q1_sec",
    "q2_sec",
    "q3_sec",
    "best_qualifying_sec",
    "completed_q3",
    "qualifying_missing",
    "qualifying_gap_to_pole",
    "qualifying_position_delta_to_grid",
    "driverAgeYears",
    "driver_home_race",
    "lat",
    "lng",
    "alt",
    "driver_prev_points",
    "driver_prev_position",
    "driver_prev_wins",
    "constructor_prev_points",
    "constructor_prev_position",
    "constructor_prev_wins",
    "driver_form_podium_rate_5",
    "driver_form_avg_finish_5",
    "driver_form_points_5",
    "constructor_form_podium_rate_5",
    "constructor_form_avg_finish_5",
    "constructor_form_points_5",
]
categorical_features = [
    "driverId",
    "constructorId",
    "circuitId",
    "driverNationality",
    "constructorNationality",
    "circuitCountryNorm",
]
feature_columns = numeric_features + categorical_features

FEATURE_GROUPS = {
    "grid": ["grid_position", "grid_zero_or_unknown"],
    "qualifying": [
        "qualifying_position",
        "q1_sec",
        "q2_sec",
        "q3_sec",
        "best_qualifying_sec",
        "completed_q3",
        "qualifying_missing",
        "qualifying_gap_to_pole",
        "qualifying_position_delta_to_grid",
    ],
    "standings": [
        "driver_prev_points",
        "driver_prev_position",
        "driver_prev_wins",
        "constructor_prev_points",
        "constructor_prev_position",
        "constructor_prev_wins",
    ],
    "form": [
        "driver_form_podium_rate_5",
        "driver_form_avg_finish_5",
        "driver_form_points_5",
        "constructor_form_podium_rate_5",
        "constructor_form_avg_finish_5",
        "constructor_form_points_5",
    ],
    "identity_context": [
        "driverId",
        "constructorId",
        "circuitId",
        "driverNationality",
        "constructorNationality",
        "circuitCountryNorm",
        "year",
        "round",
        "driverAgeYears",
        "driver_home_race",
        "lat",
        "lng",
        "alt",
    ],
}

for col in categorical_features:
    model_df[col] = model_df[col].map(lambda value: np.nan if pd.isna(value) else str(int(value)) if isinstance(value, (int, float, np.integer, np.floating)) and float(value).is_integer() else str(value))

forbidden_feature_candidates = {
    "positionOrder",
    "points",
    "laps",
    "milliseconds",
    "race_time_sec",
    "fastestLap",
    "fastestLapTime",
    "fastestLapSpeed",
    "rank",
    "statusId",
    "status",
}
leakage_overlap = forbidden_feature_candidates.intersection(feature_columns)
assert not leakage_overlap, f"Leaky feature(s) present: {leakage_overlap}"
assert model_df[["raceId", "driverId"]].duplicated().sum() == 0

modeling_data = model_df.dropna(subset=["year", "podium"]).copy()
display(
    pd.DataFrame(
        {
            "rows": [len(modeling_data)],
            "unique_races": [modeling_data["raceId"].nunique()],
            "unique_drivers": [modeling_data["driverId"].nunique()],
            "podium_rate": [modeling_data["podium"].mean()],
            "min_year": [int(modeling_data["year"].min())],
            "max_year": [int(modeling_data["year"].max())],
        }
    )
)
display(modeling_data[["raceId", "driverId", "year", "grid_position", "qualifying_position", "podium"]].head())

### Feature Selection Evidence

These plots justify the selected feature groups. Strong monotonic relationships are expected for grid and qualifying; rolling form and previous standings are leakage-safe because they are shifted to past races only.

In [ ]:
feature_eda = modeling_data.copy()
feature_eda["grid_band"] = pd.cut(
    feature_eda["grid_position"],
    bins=[0, 3, 6, 10, 15, 20, 99],
    labels=["1-3", "4-6", "7-10", "11-15", "16-20", "21+"],
    include_lowest=True,
)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

grid_rates = feature_eda.groupby("grid_band", observed=False)["podium"].mean().reset_index()
sns.barplot(data=grid_rates, x="grid_band", y="podium", ax=axes[0, 0], color="#118ab2")
axes[0, 0].set_title("Podium rate by starting-grid band")
axes[0, 0].set_ylabel("Podium rate")

qualifying_plot = feature_eda.dropna(subset=["qualifying_position"]).copy()
qualifying_plot["qualifying_band"] = pd.cut(
    qualifying_plot["qualifying_position"],
    bins=[0, 3, 6, 10, 15, 20, 99],
    labels=["1-3", "4-6", "7-10", "11-15", "16-20", "21+"],
    include_lowest=True,
)
qualifying_rates = qualifying_plot.groupby("qualifying_band", observed=False)["podium"].mean().reset_index()
sns.barplot(data=qualifying_rates, x="qualifying_band", y="podium", ax=axes[0, 1], color="#06d6a0")
axes[0, 1].set_title("Podium rate by qualifying-position band")
axes[0, 1].set_ylabel("Podium rate")

form_plot = feature_eda.dropna(subset=["driver_form_podium_rate_5"]).copy()
if form_plot["driver_form_podium_rate_5"].nunique() > 1:
    form_plot["driver_form_decile"] = pd.qcut(
        form_plot["driver_form_podium_rate_5"], q=5, duplicates="drop"
    )
    form_rates = form_plot.groupby("driver_form_decile", observed=False)["podium"].mean().reset_index()
    form_rates["driver_form_decile"] = form_rates["driver_form_decile"].astype(str)
    sns.barplot(data=form_rates, y="driver_form_decile", x="podium", ax=axes[1, 0], color="#ef476f")
    axes[1, 0].set_title("Podium rate by recent driver-form bucket")
    axes[1, 0].set_xlabel("Podium rate")
    axes[1, 0].set_ylabel("Driver form bucket")
else:
    axes[1, 0].axis("off")
    axes[1, 0].set_title("Not enough driver-form variation")

corr_cols = [
    "podium",
    "grid_position",
    "qualifying_position",
    "qualifying_gap_to_pole",
    "driver_prev_points",
    "constructor_prev_points",
    "driver_form_podium_rate_5",
    "constructor_form_podium_rate_5",
]
corr = feature_eda[corr_cols].corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=axes[1, 1])
axes[1, 1].set_title("Feature-target correlation screen")

plt.tight_layout()
plt.show()

feature_selection_note = (
    "Chosen features cover grid, qualifying, shifted championship standings, shifted recent form, and known pre-race context. "
    "Post-race result fields, pit stops, race laps, race status, current-race points, and current post-race standings are excluded."
)
display(Markdown("**Feature-selection justification:** " + feature_selection_note))

## 7. Temporal Train/Validation/Test Split

In [ ]:
def make_temporal_split(df):
    years = sorted(int(y) for y in df["year"].dropna().unique())
    preferred = {
        "train": df["year"] <= 2018,
        "validation": df["year"].between(2019, 2021),
        "test": df["year"] >= 2022,
    }
    if all(mask.sum() > 0 for mask in preferred.values()) and max(years) >= 2022:
        return preferred, "Preferred split: train <= 2018, validation 2019-2021, test >= 2022."

    train_end = max(1, int(len(years) * 0.70))
    val_end = max(train_end + 1, int(len(years) * 0.85))
    val_end = min(val_end, len(years) - 1)
    train_years = years[:train_end]
    val_years = years[train_end:val_end]
    test_years = years[val_end:]
    fallback = {
        "train": df["year"].isin(train_years),
        "validation": df["year"].isin(val_years),
        "test": df["year"].isin(test_years),
    }
    return fallback, (
        f"Fallback chronological split because the mounted dataset ends at {max(years)}: "
        f"train {min(train_years)}-{max(train_years)}, validation {min(val_years)}-{max(val_years)}, "
        f"test {min(test_years)}-{max(test_years)}."
    )

split_masks, split_note = make_temporal_split(modeling_data)
display(Markdown("**Split choice:** " + split_note + " A chronological split mirrors forecasting future races and prevents future seasons from informing earlier ones."))

X = modeling_data[feature_columns].copy()
y = modeling_data["podium"].astype(int)

X_train, y_train = X.loc[split_masks["train"]], y.loc[split_masks["train"]]
X_val, y_val = X.loc[split_masks["validation"]], y.loc[split_masks["validation"]]
X_test, y_test = X.loc[split_masks["test"]], y.loc[split_masks["test"]]

split_summary = pd.DataFrame(
    [
        {
            "split": name,
            "rows": int(mask.sum()),
            "min_year": int(modeling_data.loc[mask, "year"].min()),
            "max_year": int(modeling_data.loc[mask, "year"].max()),
            "podium_rate": float(modeling_data.loc[mask, "podium"].mean()),
        }
        for name, mask in split_masks.items()
    ]
)
display(split_summary)

## 8. Preprocessing Experiments

In [ ]:
def make_ohe():
    if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_preprocessor(num_cols, cat_cols, numeric_order="impute_scale", cat_strategy="constant"):
    transformers = []
    if num_cols:
        if numeric_order == "scale_impute":
            num_pipe = Pipeline(
                [
                    ("scaler", StandardScaler()),
                    ("imputer", SimpleImputer(strategy="median")),
                ]
            )
        else:
            num_pipe = Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            )
        transformers.append(("num", num_pipe, num_cols))

    if cat_cols:
        if cat_strategy == "most_frequent":
            cat_imputer = SimpleImputer(strategy="most_frequent")
        else:
            cat_imputer = SimpleImputer(strategy="constant", fill_value="Unknown")
        cat_pipe = Pipeline(
            [
                ("imputer", cat_imputer),
                ("onehot", make_ohe()),
            ]
        )
        transformers.append(("cat", cat_pipe, cat_cols))

    return ColumnTransformer(transformers, remainder="drop", verbose_feature_names_out=False)

def choose_threshold(y_true, proba):
    if pd.Series(y_true).nunique() < 2:
        return 0.5
    thresholds = np.linspace(0.05, 0.95, 181)
    f1s = [f1_score(y_true, proba >= t, zero_division=0) for t in thresholds]
    return float(thresholds[int(np.argmax(f1s))])

def score_predictions(y_true, proba, threshold):
    y_pred = (np.asarray(proba) >= threshold).astype(int)
    out = {
        "roc_auc": np.nan,
        "pr_auc": np.nan,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "threshold": threshold,
    }
    if pd.Series(y_true).nunique() == 2:
        out["roc_auc"] = roc_auc_score(y_true, proba)
        out["pr_auc"] = average_precision_score(y_true, proba)
    return out

def make_logistic_pipeline(num_cols, cat_cols, numeric_order="impute_scale", cat_strategy="constant"):
    return Pipeline(
        [
            ("preprocess", make_preprocessor(num_cols, cat_cols, numeric_order, cat_strategy)),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    solver="lbfgs",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

preprocessing_experiment_specs = [
    {
        "experiment": "A: sentinel-to-NaN, median impute numeric then scale, Unknown category then one-hot",
        "numeric_order": "impute_scale",
        "cat_strategy": "constant",
    },
    {
        "experiment": "B: sentinel-to-NaN, scale numeric then median impute, Unknown category then one-hot",
        "numeric_order": "scale_impute",
        "cat_strategy": "constant",
    },
    {
        "experiment": "C: sentinel-to-NaN, median impute numeric then scale, most-frequent category then one-hot",
        "numeric_order": "impute_scale",
        "cat_strategy": "most_frequent",
    },
]

preprocessing_results = []
for spec in preprocessing_experiment_specs:
    try:
        pipe = make_logistic_pipeline(
            numeric_features,
            categorical_features,
            numeric_order=spec["numeric_order"],
            cat_strategy=spec["cat_strategy"],
        )
        pipe.fit(X_train, y_train)
        val_proba = pipe.predict_proba(X_val)[:, 1]
        threshold = choose_threshold(y_val, val_proba)
        metrics = score_predictions(y_val, val_proba, threshold)
        preprocessing_results.append({**spec, **metrics})
    except Exception as exc:
        preprocessing_results.append({**spec, "error": repr(exc)})

preprocessing_results_df = pd.DataFrame(preprocessing_results)
display(preprocessing_results_df.sort_values("pr_auc", ascending=False, na_position="last"))

best_preprocessing_row = preprocessing_results_df.dropna(subset=["pr_auc"]).sort_values("pr_auc", ascending=False).iloc[0]
BEST_NUMERIC_ORDER = best_preprocessing_row["numeric_order"]
BEST_CAT_STRATEGY = best_preprocessing_row["cat_strategy"]
display(
    Markdown(
        f"**Selected preprocessing:** `{BEST_NUMERIC_ORDER}` for numeric features and `{BEST_CAT_STRATEGY}` for categorical features, "
        "chosen by validation PR-AUC using the baseline logistic model."
    )
)

## 9. Predictive Modeling

In [ ]:
fitted_models = {}
prediction_cache = {}
model_metric_rows = []

def add_model_scores(model_name, kind, model_object, threshold, proba_by_split, extra=None):
    for split_name, y_split in [("train", y_train), ("validation", y_val), ("test", y_test)]:
        metrics = score_predictions(y_split, proba_by_split[split_name], threshold)
        row = {"model": model_name, "split": split_name, "kind": kind, **metrics}
        model_metric_rows.append(row)
    prediction_cache[model_name] = proba_by_split
    fitted_models[model_name] = {"kind": kind, "model": model_object, "threshold": threshold, **(extra or {})}

statistical_estimators = {
    "Logistic Regression": LogisticRegression(
        max_iter=2500,
        class_weight="balanced",
        solver="lbfgs",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=220,
        min_samples_leaf=4,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

for model_name, estimator in statistical_estimators.items():
    pipe = Pipeline(
        [
            ("preprocess", make_preprocessor(numeric_features, categorical_features, BEST_NUMERIC_ORDER, BEST_CAT_STRATEGY)),
            ("model", estimator),
        ]
    )
    pipe.fit(X_train, y_train)
    proba_by_split = {
        "train": pipe.predict_proba(X_train)[:, 1],
        "validation": pipe.predict_proba(X_val)[:, 1],
        "test": pipe.predict_proba(X_test)[:, 1],
    }
    threshold = choose_threshold(y_val, proba_by_split["validation"])
    add_model_scores(model_name, "pipeline", pipe, threshold, proba_by_split)

# Shallow neural net. TensorFlow/Keras is used on Kaggle when available; sklearn MLP is a fallback.
nn_name = "Shallow Neural Net"
try:
    import tensorflow as tf

    tf.keras.utils.set_random_seed(RANDOM_STATE)
    nn_preprocessor = make_preprocessor(numeric_features, categorical_features, BEST_NUMERIC_ORDER, BEST_CAT_STRATEGY)
    X_train_nn = nn_preprocessor.fit_transform(X_train)
    X_val_nn = nn_preprocessor.transform(X_val)
    X_test_nn = nn_preprocessor.transform(X_test)

    classes = np.array([0, 1])
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight = {int(cls): float(weight) for cls, weight in zip(classes, weights)}

    nn_model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(X_train_nn.shape[1],)),
            tf.keras.layers.Dense(64, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
            tf.keras.layers.Dropout(0.20),
            tf.keras.layers.Dense(32, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ]
    )
    nn_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(curve="PR", name="pr_auc"), tf.keras.metrics.AUC(curve="ROC", name="roc_auc")],
    )
    history = nn_model.fit(
        X_train_nn,
        y_train,
        validation_data=(X_val_nn, y_val),
        epochs=50,
        batch_size=256,
        verbose=0,
        class_weight=class_weight,
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=6, restore_best_weights=True)],
    )
    proba_by_split = {
        "train": nn_model.predict(X_train_nn, verbose=0).ravel(),
        "validation": nn_model.predict(X_val_nn, verbose=0).ravel(),
        "test": nn_model.predict(X_test_nn, verbose=0).ravel(),
    }
    threshold = choose_threshold(y_val, proba_by_split["validation"])
    add_model_scores(
        nn_name,
        "tensorflow",
        nn_model,
        threshold,
        proba_by_split,
        extra={"preprocessor": nn_preprocessor, "history": history},
    )
except Exception as exc:
    print(f"TensorFlow/Keras path unavailable ({exc}). Falling back to sklearn MLPClassifier.")
    from sklearn.neural_network import MLPClassifier

    mlp_pipe = Pipeline(
        [
            ("preprocess", make_preprocessor(numeric_features, categorical_features, BEST_NUMERIC_ORDER, BEST_CAT_STRATEGY)),
            (
                "model",
                MLPClassifier(
                    hidden_layer_sizes=(64, 32),
                    activation="relu",
                    alpha=1e-4,
                    batch_size=256,
                    learning_rate_init=1e-3,
                    early_stopping=True,
                    validation_fraction=0.15,
                    max_iter=120,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )
    mlp_pipe.fit(X_train, y_train)
    proba_by_split = {
        "train": mlp_pipe.predict_proba(X_train)[:, 1],
        "validation": mlp_pipe.predict_proba(X_val)[:, 1],
        "test": mlp_pipe.predict_proba(X_test)[:, 1],
    }
    threshold = choose_threshold(y_val, proba_by_split["validation"])
    add_model_scores(nn_name, "pipeline", mlp_pipe, threshold, proba_by_split)

comparison_df = pd.DataFrame(model_metric_rows)
metric_order = ["roc_auc", "pr_auc", "precision", "recall", "f1", "threshold"]
display(comparison_df[["model", "split"] + metric_order].sort_values(["split", "pr_auc"], ascending=[True, False]))

validation_scores = comparison_df.query("split == 'validation'").sort_values("pr_auc", ascending=False)
PRODUCTION_MODEL_NAME = validation_scores.iloc[0]["model"]
display(
    Markdown(
        f"**Primary metric:** PR-AUC, because podiums are the minority class and ranking likely podium finishers is the key task. "
        f"**Secondary metric:** F1 at a validation-selected threshold. Selected model by validation PR-AUC: **{PRODUCTION_MODEL_NAME}**."
    )
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

compare_plot = comparison_df[comparison_df["split"].isin(["validation", "test"])]
sns.barplot(data=compare_plot, x="model", y="pr_auc", hue="split", ax=axes[0])
axes[0].set_title("PR-AUC by model")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=compare_plot, x="model", y="f1", hue="split", ax=axes[1])
axes[1].set_title("F1 by model")
axes[1].tick_params(axis="x", rotation=20)

best_info = fitted_models[PRODUCTION_MODEL_NAME]
best_threshold = best_info["threshold"]
best_test_proba = prediction_cache[PRODUCTION_MODEL_NAME]["test"]
best_test_pred = (best_test_proba >= best_threshold).astype(int)
cm = confusion_matrix(y_test, best_test_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[2])
axes[2].set_title(f"Test confusion matrix: {PRODUCTION_MODEL_NAME}")
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("Actual")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for model_name, proba_by_split in prediction_cache.items():
    if y_test.nunique() == 2:
        fpr, tpr, _ = roc_curve(y_test, proba_by_split["test"])
        precision, recall, _ = precision_recall_curve(y_test, proba_by_split["test"])
        axes[0].plot(fpr, tpr, label=model_name)
        axes[1].plot(recall, precision, label=model_name)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
axes[0].set_title("Test ROC curves")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].legend()
axes[1].set_title("Test precision-recall curves")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()
plt.tight_layout()
plt.show()

## 10. Feature-Group Ablation

In [ ]:
full_logit_pr_auc = float(
    comparison_df.query("model == 'Logistic Regression' and split == 'validation'")["pr_auc"].iloc[0]
)

ablation_rows = []
for group_name, remove_cols in FEATURE_GROUPS.items():
    remove_cols = set(remove_cols)
    ablation_numeric = [col for col in numeric_features if col not in remove_cols]
    ablation_categorical = [col for col in categorical_features if col not in remove_cols]
    try:
        pipe = make_logistic_pipeline(
            ablation_numeric,
            ablation_categorical,
            numeric_order=BEST_NUMERIC_ORDER,
            cat_strategy=BEST_CAT_STRATEGY,
        )
        pipe.fit(X_train[ablation_numeric + ablation_categorical], y_train)
        val_proba = pipe.predict_proba(X_val[ablation_numeric + ablation_categorical])[:, 1]
        test_proba = pipe.predict_proba(X_test[ablation_numeric + ablation_categorical])[:, 1]
        threshold = choose_threshold(y_val, val_proba)
        val_metrics = score_predictions(y_val, val_proba, threshold)
        test_metrics = score_predictions(y_test, test_proba, threshold)
        ablation_rows.append(
            {
                "dropped_feature_group": group_name,
                "remaining_features": len(ablation_numeric) + len(ablation_categorical),
                "validation_pr_auc": val_metrics["pr_auc"],
                "validation_f1": val_metrics["f1"],
                "test_pr_auc": test_metrics["pr_auc"],
                "test_f1": test_metrics["f1"],
                "validation_pr_auc_delta_vs_full_logit": val_metrics["pr_auc"] - full_logit_pr_auc,
            }
        )
    except Exception as exc:
        ablation_rows.append({"dropped_feature_group": group_name, "error": repr(exc)})

ablation_df = pd.DataFrame(ablation_rows).sort_values("validation_pr_auc_delta_vs_full_logit")
display(ablation_df)

fig, ax = plt.subplots(figsize=(12, 5))
if "validation_pr_auc_delta_vs_full_logit" in ablation_df:
    sns.barplot(
        data=ablation_df,
        x="validation_pr_auc_delta_vs_full_logit",
        y="dropped_feature_group",
        ax=ax,
        color="#b85042",
    )
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title("Ablation impact: validation PR-AUC change vs full logistic baseline")
    ax.set_xlabel("Validation PR-AUC delta")
    ax.set_ylabel("Dropped group")
else:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 11. Model Explainability (XAI)

The explanations below are model-reliance explanations, not causal effects. If a feature is important to the model, that means the model used it for prediction; it does not prove the feature causes podium finishes.

In [ ]:
xai_model_name = "Random Forest" if "Random Forest" in fitted_models else PRODUCTION_MODEL_NAME
xai_info = fitted_models[xai_model_name]
xai_model = xai_info["model"]

if xai_info["kind"] == "pipeline":
    perm_result = permutation_importance(
        xai_model,
        X_val,
        y_val,
        scoring="average_precision",
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    permutation_df = (
        pd.DataFrame(
            {
                "feature": feature_columns,
                "importance_mean": perm_result.importances_mean,
                "importance_std": perm_result.importances_std,
            }
        )
        .sort_values("importance_mean", ascending=False)
        .reset_index(drop=True)
    )
    display(permutation_df.head(20))

    fig, ax = plt.subplots(figsize=(12, 7))
    sns.barplot(data=permutation_df.head(20), x="importance_mean", y="feature", ax=ax, color="#118ab2")
    ax.set_title(f"Permutation importance on validation PR-AUC: {xai_model_name}")
    ax.set_xlabel("Mean PR-AUC drop when permuted")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()
else:
    permutation_df = pd.DataFrame()
    display(Markdown("Permutation importance is skipped for the TensorFlow object; SHAP/LIME use the Random Forest when available."))

In [ ]:
def import_or_install(package_name):
    try:
        return __import__(package_name)
    except ImportError:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
            return __import__(package_name)
        except Exception as exc:
            print(f"Could not import or install {package_name}: {exc}")
            return None

def class1_shap_values(raw_values):
    if isinstance(raw_values, list):
        return np.asarray(raw_values[1])
    arr = np.asarray(raw_values)
    if arr.ndim == 3:
        if arr.shape[-1] == 2:
            return arr[:, :, 1]
        if arr.shape[0] == 2:
            return arr[1, :, :]
    return arr

def class1_expected_value(explainer):
    value = explainer.expected_value
    arr = np.asarray(value)
    if arr.ndim > 0 and len(arr) > 1:
        return arr[1]
    return float(arr)

shap = import_or_install("shap")
shap_summary_df = pd.DataFrame()
local_shap_table = pd.DataFrame()

if shap is not None and "Random Forest" in fitted_models:
    rf_pipe = fitted_models["Random Forest"]["model"]
    rf_preprocessor = rf_pipe.named_steps["preprocess"]
    rf_estimator = rf_pipe.named_steps["model"]

    shap_raw = X_val.sample(n=min(300, len(X_val)), random_state=RANDOM_STATE)
    shap_array = rf_preprocessor.transform(shap_raw)
    feature_names_out = list(rf_preprocessor.get_feature_names_out())
    shap_frame = pd.DataFrame(shap_array, columns=feature_names_out, index=shap_raw.index)

    tree_explainer = shap.TreeExplainer(rf_estimator)
    shap_values_raw = tree_explainer.shap_values(shap_frame)
    shap_values = class1_shap_values(shap_values_raw)

    shap_summary_df = (
        pd.DataFrame({"feature": feature_names_out, "mean_abs_shap": np.abs(shap_values).mean(axis=0)})
        .sort_values("mean_abs_shap", ascending=False)
        .reset_index(drop=True)
    )
    display(shap_summary_df.head(20))

    shap.summary_plot(shap_values, shap_frame, max_display=20, show=False)
    plt.title("Global SHAP summary for podium class")
    plt.tight_layout()
    plt.show()

    local_raw_index = int(np.argmax(rf_pipe.predict_proba(X_val)[:, 1]))
    local_raw = X_val.iloc[[local_raw_index]]
    local_array = rf_preprocessor.transform(local_raw)
    local_frame = pd.DataFrame(local_array, columns=feature_names_out)
    local_values = class1_shap_values(tree_explainer.shap_values(local_frame))[0]
    base_value = class1_expected_value(tree_explainer)

    local_shap_table = (
        pd.DataFrame(
            {
                "feature": feature_names_out,
                "feature_value": local_frame.iloc[0].values,
                "shap_value": local_values,
                "abs_shap_value": np.abs(local_values),
            }
        )
        .sort_values("abs_shap_value", ascending=False)
        .head(15)
    )
    display(local_shap_table)

    try:
        local_explanation = shap.Explanation(
            values=local_values,
            base_values=base_value,
            data=local_frame.iloc[0].values,
            feature_names=feature_names_out,
        )
        shap.plots.waterfall(local_explanation, max_display=15, show=False)
        plt.title("Local SHAP explanation for one validation prediction")
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print(f"Waterfall plot unavailable: {exc}")
else:
    display(Markdown("SHAP could not be run in this environment. On Kaggle, `shap` is usually available or installable."))

In [ ]:
lime_module = import_or_install("lime")

if "Random Forest" in fitted_models:
    lime_pipe = fitted_models["Random Forest"]["model"]
else:
    lime_pipe = fitted_models[PRODUCTION_MODEL_NAME]["model"]

if hasattr(lime_pipe, "named_steps"):
    lime_preprocessor = lime_pipe.named_steps["preprocess"]
    lime_estimator = lime_pipe.named_steps["model"]
    lime_feature_names = list(lime_preprocessor.get_feature_names_out())
    X_train_lime = lime_preprocessor.transform(X_train)
    X_val_lime = lime_preprocessor.transform(X_val)
    local_position = int(np.argmax(lime_pipe.predict_proba(X_val)[:, 1]))

    if lime_module is not None:
        from lime.lime_tabular import LimeTabularExplainer

        lime_explainer = LimeTabularExplainer(
            np.asarray(X_train_lime),
            feature_names=lime_feature_names,
            class_names=["not podium", "podium"],
            mode="classification",
            discretize_continuous=True,
            random_state=RANDOM_STATE,
        )
        lime_exp = lime_explainer.explain_instance(
            np.asarray(X_val_lime[local_position]),
            lime_estimator.predict_proba,
            num_features=15,
        )
        lime_table = pd.DataFrame(lime_exp.as_list(), columns=["feature_rule", "local_weight"])
        display(lime_table)
        fig = lime_exp.as_pyplot_figure()
        plt.title("LIME local explanation for one validation prediction")
        plt.tight_layout()
        plt.show()
    else:
        # Lightweight LIME-style fallback: fit a distance-weighted local ridge surrogate
        # around the selected transformed validation row.
        rng = np.random.default_rng(RANDOM_STATE)
        sample_size = min(1200, X_train_lime.shape[0])
        sample_indices = rng.choice(X_train_lime.shape[0], size=sample_size, replace=False)
        local_center = np.asarray(X_val_lime[local_position]).reshape(1, -1)
        neighborhood = np.asarray(X_train_lime)[sample_indices]
        distances = np.linalg.norm(neighborhood - local_center, axis=1)
        kernel_width = np.sqrt(neighborhood.shape[1]) * 0.75
        weights = np.exp(-(distances**2) / (kernel_width**2 + 1e-9))
        local_targets = lime_estimator.predict_proba(neighborhood)[:, 1]
        surrogate = Ridge(alpha=1.0)
        surrogate.fit(neighborhood, local_targets, sample_weight=weights)
        lime_table = (
            pd.DataFrame({"feature_rule": lime_feature_names, "local_weight": surrogate.coef_})
            .assign(abs_weight=lambda d: d["local_weight"].abs())
            .sort_values("abs_weight", ascending=False)
            .head(15)
        )
        display(lime_table)
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.barplot(data=lime_table, x="local_weight", y="feature_rule", ax=ax, color="#06d6a0")
        ax.axvline(0, color="black", linewidth=1)
        ax.set_title("LIME-style local surrogate explanation")
        ax.set_xlabel("Local surrogate weight")
        ax.set_ylabel("")
        plt.tight_layout()
        plt.show()
else:
    display(Markdown("Local LIME explanation uses a pipeline model; the selected production model is not a pipeline."))

## 12. Inference Function

In [ ]:
def predict_proba_from_model_info(model_info, frame):
    if model_info["kind"] == "tensorflow":
        arr = model_info["preprocessor"].transform(frame)
        return model_info["model"].predict(arr, verbose=0).ravel()
    return model_info["model"].predict_proba(frame)[:, 1]

def predict_podium(record, model_name=None, threshold=None):
    selected_name = model_name or PRODUCTION_MODEL_NAME
    model_info = fitted_models[selected_name]
    selected_threshold = model_info["threshold"] if threshold is None else threshold

    if isinstance(record, pd.Series):
        frame = record.to_frame().T
    elif isinstance(record, pd.DataFrame):
        frame = record.copy()
    else:
        frame = pd.DataFrame([record])

    for col in feature_columns:
        if col not in frame.columns:
            frame[col] = np.nan
    frame = frame[feature_columns]

    proba = predict_proba_from_model_info(model_info, frame)
    return pd.DataFrame(
        {
            "model": selected_name,
            "podium_probability": proba,
            "threshold": selected_threshold,
            "podium_prediction": proba >= selected_threshold,
        }
    )

demo_source = X_test if len(X_test) else X_val
complete_record = demo_source.iloc[0].to_dict()
partial_keys = [
    "year",
    "round",
    "grid_position",
    "qualifying_position",
    "constructorId",
    "circuitId",
    "driver_form_podium_rate_5",
    "constructor_form_podium_rate_5",
]
partial_record = {key: complete_record.get(key, np.nan) for key in partial_keys if key in feature_columns}

display(Markdown("Complete-record inference:"))
display(predict_podium(complete_record))
display(Markdown("Partially-missing-record inference. Missing columns are handled by the trained preprocessing pipeline:"))
display(predict_podium(partial_record))

## 13. Structured Analytical Report

In [ ]:
def safe_metric(model_name, split, metric):
    rows = comparison_df.query("model == @model_name and split == @split")
    if rows.empty:
        return np.nan
    return float(rows.iloc[0][metric])

best_test_pr = safe_metric(PRODUCTION_MODEL_NAME, "test", "pr_auc")
best_test_f1 = safe_metric(PRODUCTION_MODEL_NAME, "test", "f1")

ablation_text = "Ablation results were not available."
if "ablation_df" in globals() and not ablation_df.empty and "validation_pr_auc_delta_vs_full_logit" in ablation_df:
    strongest_group = ablation_df.sort_values("validation_pr_auc_delta_vs_full_logit").iloc[0]
    ablation_text = (
        f"The largest validation PR-AUC drop came from removing `{strongest_group['dropped_feature_group']}` "
        f"({strongest_group['validation_pr_auc_delta_vs_full_logit']:.4f} vs the full logistic baseline)."
    )

report_text = f'''
# Formula 1 Podium Prediction Companion - Milestone 1 Report

## Data audit and cleaning
The raw dataset was loaded as immutable tables, literal `\\\\N` and blank sentinels were converted to missing values in copies, dates were parsed as datetimes, and lap/qualifying/pit durations were converted into seconds where useful. Primary-key and foreign-key audits are shown in the notebook. Shared-drive duplicate `(raceId, driverId)` rows were handled by keeping the row with the most laps, then best classified finish, then lowest `resultId`.

## Data-engineering answers
1. Front-row conversion: {q1_report_sentence}
2. Home race effect: {q2_report_sentence}
3. Mechanical retirements: {q3_report_sentence}

## Feature set and leakage control
Features include starting grid, qualifying position and times, qualifying-to-grid delta, shifted prior race standings, shifted recent driver and constructor form, driver age, circuit geography, home-race indicator, and pre-race identity/context. Excluded fields include current-race finishing position except for the target, current race points, race laps, status, pit stops, fastest laps, race time, and post-race standings.

## Split and metrics
{split_note} PR-AUC is the primary metric because podiums are rare relative to non-podium finishes. F1 is secondary and uses thresholds selected on validation data only. ROC-AUC, PR-AUC, precision, recall, and F1 are reported for train, validation, and test splits.

## Model comparison
The selected model by validation PR-AUC is `{PRODUCTION_MODEL_NAME}`. On the unseen test split it reached PR-AUC `{best_test_pr:.4f}` and F1 `{best_test_f1:.4f}` at its validation-selected threshold.

## Experiments
Preprocessing-order experiments compare numeric imputation/scaling order and categorical missing-value handling. Feature-group ablation retrains the baseline after removing one feature group at a time. {ablation_text}

## XAI
Global explanations include permutation importance and SHAP. Local explanations include local SHAP and LIME, with a local surrogate fallback if the external LIME package is unavailable. These explanations show model reliance only; they are not causal claims.

## Known limitations
Historical qualifying coverage is incomplete, especially in older seasons. The home-race analysis depends on a simplified nationality-to-country mapping. Mechanical-retirement classification uses status-label keyword rules and can miss ambiguous failures. Era changes mean older data may not fully represent current F1. Driver and constructor identifiers can encode historical strength, so future-newcomer generalization should be monitored.
'''
report_text = dedent(report_text).strip()
display(Markdown(report_text))
Path("MS1_report.md").write_text(report_text, encoding="utf-8")
print("Wrote MS1_report.md in the current working directory.")

## Checklist Mapping

- EDA before and after cleaning: Sections 2 and 3.
- Sentinel handling, semantic typing, primary-key and foreign-key audits: Section 3.
- Unique driver-race modeling grain and shared-drive duplicate policy: Section 4.
- Three joined data-engineering questions with plots and written answers: Section 5.
- Leakage-safe feature engineering, feature-selection evidence, and temporal split: Sections 6 and 7.
- Preprocessing-order experiments: Section 8.
- Logistic regression, random forest, and shallow neural net comparison: Section 9.
- Feature-group ablation: Section 10.
- Permutation importance, SHAP, and local explanation: Section 11.
- Inference function with complete and partially missing records: Section 12.
- Structured analytical report: Section 13.